In [1]:
import math
import traceback
from steel_lib.si_units import si
from steelpy import aisc

from steel_lib.data_models import (
    Plate,
    ConnectionFactory,
    ConnectionComponent,
    DesignLoads,
    AppliedLoads,
)
from steel_lib.materials import MATERIALS, BOLT_GRADES, WELD_ELECTRODES
from steel_lib.member_factory import MemberFactory
from steel_lib.calculations import (
    BoltShearCalculator,
    BlockShearCalculator,
    ConnectionCapacityCalculator,
    TensileYieldingCalculator,
    TensileRuptureCalculator,
    TensileYieldWhitmore,
    CompressionBucklingCalculator,
    UFMCalculator,
    PlateTensileYieldingCalculator,
    WebLocalYieldingCalculator,
    WebLocalCrippingCalculator,
    ShearYieldingCalculator,
    PryingActionCalculator,
    AdmissableDistortionForces,
    WeldCalculator
)

In [2]:
beam = MemberFactory.create_steelpy_member(
    section_class=aisc.W_shapes,
    section_name="W21X83",
    material=MATERIALS["a992"],
    shape_type="W"
)

support = MemberFactory.create_steelpy_member(
    section_class=aisc.W_shapes,
    section_name="W14X90",
    material=MATERIALS["a992"],
    shape_type="W"
)

bracing = MemberFactory.create_steelpy_member(
    section_class=aisc.L_shapes,
    section_name="L8X6X1",
    material=MATERIALS["a36"],
    shape_type="L",
    loading_condition=2,  # Assuming this is a bracing member
)

# End Plate for Column Connection
end_plate_column = Plate(
    t=5/8 * si.inch,
    material=MATERIALS["a572_gr50"],
    width=10 * si.inch,
)
end_plate_beam = Plate(
    t=3/4 * si.inch,
    material=MATERIALS["a572_gr50"],
    width=10 * si.inch,
    length = beam.d + 1 * si.inch,  # Assuming the end plate length is equal to the beam depth plus some extra
)
    # Gusset Plate for Bracing Connection
gusset_plate = Plate(
        t=1 * si.inch,
        material=MATERIALS["a572_gr50"],
        clipping=3/4 * si.inch,
    )


In [ ]:
brace_gusset_loads = DesignLoads(Pu = 840 * si.kip, Vu = 0 * si.kip, Aub = 0 * si.kip)

brace_gusset_connection = ConnectionFactory.create_bolted_connection(
        member_a=bracing,
        member_b=gusset_plate,
        component_a=ConnectionComponent.TOTAL,
        row_spacing=3.0 * si.inch,
        column_spacing=3.0 * si.inch,
        n_rows=2,
        n_columns=7,
        edge_distance_vertical=2 * si.inch,
        edge_distance_horizontal=1.5 * si.inch,
        bolt_diameter=7/8 * si.inch,
        bolt_grade=BOLT_GRADES["a325_x"],
        material=MATERIALS["a572_gr50"],
        angle=47.2 * math.pi / 180
    )
brace_gusset_tensile_yielding = TensileYieldingCalculator(endpoint=brace_gusset_connection.member_a, connection=brace_gusset_connection,load=brace_gusset_loads)
brace_gusset_tensile_yielding.check_dcr(debug=True)
brace_gusset_tensile_rupture = TensileRuptureCalculator(endpoint=brace_gusset_connection.member_a, connection=brace_gusset_connection,load=brace_gusset_loads)
brace_gusset_tensile_rupture.check_dcr(debug=True)
brace_gusset_connection_strength = ConnectionCapacityCalculator(endpoint=brace_gusset_connection.member_b, connection=brace_gusset_connection,loads=brace_gusset_loads)
brace_gusset_connection_strength.check_dcr(debug = True,number_of_shear_planes=2)
brace_gusset_block_shear = BlockShearCalculator(endpoint=brace_gusset_connection.member_a, connection=brace_gusset_connection,loads=brace_gusset_loads, loading_condition=2)
brace_gusset_block_shear.check_dcr(debug=True)


# Gusset Plate Check
gusset_plate_block_shear = BlockShearCalculator(
    endpoint=brace_gusset_connection.member_b,
    connection=brace_gusset_connection,
    loads=brace_gusset_loads,)
gusset_plate_block_shear.check_dcr(debug=True)
gusset_plate_connection_strength = ConnectionCapacityCalculator(
    endpoint=brace_gusset_connection.member_b,
    connection=brace_gusset_connection,
    loads=brace_gusset_loads,
)
gusset_plate_connection_strength.check_dcr(number_of_shear_planes = 2 , debug=True)
gusset_plate_whitmore = TensileYieldWhitmore(
    endpoint=brace_gusset_connection.member_b,
    connection=brace_gusset_connection)
gusset_plate_whitmore.check_dcr(debug=True)




--- DEBUG: Tensile Yielding (TOTAL) ---
  Inputs:
    Yield Strength (Fy)                : 36.000 ksi
    Applicable Gross Area (Ag) for TOTAL: 13.100 inch²
    Loading Condition                  : 2
    Resistance Factor (phi)            : 0.9000
  Calculations:
    Nominal Strength (Rn = Fy * Ag)    : 471.600 kip
  Output:
                                         --------------------
    Design Strength (phiRn)            : 848.880 kip
--- END DEBUG: Tensile Yielding (TOTAL) ---

--- DEBUG: Tensile Rupture (TOTAL) ---
  Inputs:
    Ultimate Strength (Fu)             : 58.000 ksi
    Applicable Thickness (t)           : 1.000 inch
    Gross Area (Ag)                    : 13.100 inch²
    Bolt Diameter (d_bolt)             : 0.875 inch
    Bolt Columns (N_c)                 : 7
    Bolt Rows (n_rows)                 : 2
    Column Spacing (S_c)               : 3.000 inch
    Loading Condition                  : 2
    Resistance Factor (phi)            : 0.7500
    Eccentricity (x_bar)